In [ ]:
# Environment Setup & Dependency Installation

import sys
import os
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

# Detect Environment
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()
PYTHON_VERSION = sys.version.split()[0]

print(f"Environment Detected: {'Google Colab' if IS_COLAB else 'Local VS Code / Jupyter'}")
print(f"Kernel Python Version: {PYTHON_VERSION}\n")

# Packages to install
PACKAGES = "transformers datasets evaluate nltk seaborn matplotlib accelerate numpy pandas torch scikit-learn"

if IS_COLAB:
    # --- Colab-Specific Installation ---
    try:
        import pyarrow
        import datasets
        import evaluate
        import sklearn
        NEEDS_INSTALL = False
    except (ImportError, ValueError):
        NEEDS_INSTALL = True

    if NEEDS_INSTALL:
        print("Installing/Updating core dependencies...")
        !pip install -q pyarrow
        !pip install -q {PACKAGES}
        print("✓ Dependencies installed/updated!")
        print("⚠️  Restarting Runtime...")
        print("📝 Re-run the Notebook...")
        time.sleep(2)
        os.kill(os.getpid(), 9)
    else:
        print("✓ Dependencies already installed!")
else:
    # --- Local Jupyter/VS Code Installation ---
    print("Installing/Updating core dependencies")
    try:
        if 'get_ipython' in globals():
            cell_content = f"%%capture\n%pip install -U {PACKAGES}"
            get_ipython().run_cell(cell_content)
            print("✓ Dependencies installed/updated!")
        else:
            print("⚠️  Installation failed: Not running in interactive environment. Run manually.")
    except NameError:
        print("⚠️  Installation failed: Not running in a standard Jupyter kernel. Run manually.")

Environment Detected: Local VS Code / Jupyter
Kernel Python Version: 3.12.10

Installing/Updating core dependencies
✓ Dependencies installed/updated!


In [2]:
# Importing Libraries

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from datasets import load_from_disk
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Only needed for Colab-specific functions
if IS_COLAB:
    from IPython.display import display, HTML
    import shutil  # For zipping files
    from google.colab import files  # For downloading files

print("✅ All Libraries Imported Successfully")

c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All Libraries Imported Successfully


In [3]:
# Setup Training Paths and Configuration

print("\n Setting up training environment...\n")

# Core constants
MODEL_CKPT = "distilbert-base-uncased"
CLASS_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
model_folder_name = "Distilbert"  # Consistent capitalization

# Define paths based on environment
if IS_COLAB:
    DATASET_PATH = "/content/data"
    MODEL_BASE_DIR = f"/content/models/{model_folder_name}"
    OUTPUT_DIR = f"{MODEL_BASE_DIR}/distilbert-checkpoints"
    FINAL_MODEL_DIR = f"{MODEL_BASE_DIR}/distilbert-final"
else:
    # Local paths - notebook is in notebooks/ folder
    DATASET_PATH = "../data"
    MODEL_BASE_DIR = f"../models/{model_folder_name}"
    OUTPUT_DIR = f"{MODEL_BASE_DIR}/distilbert-checkpoints"
    FINAL_MODEL_DIR = f"{MODEL_BASE_DIR}/distilbert-final"

# Create directories
os.makedirs(MODEL_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify dataset exists
if not os.path.exists(DATASET_PATH):
    print(f"Current directory: {os.getcwd()}")
    print(f"Looking for data at: {os.path.abspath(DATASET_PATH)}")
    raise FileNotFoundError(f"❌ Dataset not found at: {DATASET_PATH}\n   Please run Notebook 1 (EDA & Preprocessing) first.")

print(f"✅ Dataset path: {os.path.abspath(DATASET_PATH)}")
print(f"✅ Model base dir: {os.path.abspath(MODEL_BASE_DIR)}")
print(f"✅ Checkpoints dir: {os.path.abspath(OUTPUT_DIR)}")
print(f"✅ Final model dir: {os.path.abspath(FINAL_MODEL_DIR)}\n")


 Setting up training environment...

✅ Dataset path: c:\Users\joaqu\AI-News-Classification\data
✅ Model base dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert
✅ Checkpoints dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-checkpoints
✅ Final model dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-final



In [4]:
# Load Dataset

print("📂 Loading tokenized dataset...\n")

tokenized = load_from_disk(DATASET_PATH)
tokenized_train = tokenized["train"]
tokenized_test = tokenized["test"]

print("✅ Dataset Loaded Successfully")
print(f"Train samples: {len(tokenized_train):,}")
print(f"Test samples:  {len(tokenized_test):,}")
print(f"Features: {list(tokenized_train.features.keys())}")

📂 Loading tokenized dataset...

✅ Dataset Loaded Successfully
Train samples: 2,000
Test samples:  500
Features: ['text', 'label', 'input_ids', 'attention_mask']


In [5]:
# Model Setup and Training Configuration

# Load Model and Tokenizer
print("🤖 Loading model and tokenizer...\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(CLASS_NAMES),
    id2label=CLASS_NAMES,
    label2id={v: k for k, v in CLASS_NAMES.items()}
)

print(f"✅ Model loaded: {MODEL_CKPT}")
print(f"✅ Tokenizer loaded")
print(f"📊 Number of classes: {len(CLASS_NAMES)}")
print(f"🏷️  Classes: {list(CLASS_NAMES.values())}\n")

# Define Metrics
print("📈 Setting up evaluation metrics...\n")

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy and F1 score during evaluation"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

print("✅ Metrics configured: Accuracy & F1 (macro)\n")

# Training Configuration
print("⚙️  Configuring training parameters...\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=20,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    greater_is_better=True,
    report_to="none",
    save_total_limit=2
)

print("✅ Training Configuration:")
print(f"  • Epochs: {training_args.num_train_epochs}")
print(f"  • Learning rate: {training_args.learning_rate}")
print(f"  • Batch size: {training_args.per_device_train_batch_size}")
print(f"  • Weight decay: {training_args.weight_decay}")
print(f"  • Checkpoints: {os.path.abspath(OUTPUT_DIR)}\n")

# Initialize Trainer
print("🏗️  Initializing Trainer...\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("✅ Trainer initialized and ready!\n")

🤖 Loading model and tokenizer...



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded: distilbert-base-uncased
✅ Tokenizer loaded
📊 Number of classes: 4
🏷️  Classes: ['World', 'Sports', 'Business', 'Sci/Tech']

📈 Setting up evaluation metrics...

✅ Metrics configured: Accuracy & F1 (macro)

⚙️  Configuring training parameters...

✅ Training Configuration:
  • Epochs: 3
  • Learning rate: 2e-05
  • Batch size: 16
  • Weight decay: 0.01
  • Checkpoints: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-checkpoints

🏗️  Initializing Trainer...

✅ Trainer initialized and ready!



C:\Users\joaqu\AppData\Local\Temp\ipykernel_22940\3652518343.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
#TRAIN MODEL & SAVE FINAL MODEL

print("\nSTARTING TRAINING")
trainer.train()
print("TRAINING COMPLETE!")

# Evaluation (Optional but good practice)
print("\nEvaluating final model on test set...\n")

eval_results = trainer.evaluate()

print("\nFinal Test Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key:.<30} {value:.4f}")
    else:
        print(f"{key:.<30} {value}")

# Saving Model
print(f"\nSaving final trained model to: {os.path.abspath(FINAL_MODEL_DIR)}\n")

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# Save the best model (using the global FINAL_MODEL_DIR)
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR) 

print(f"Model saved to: {os.path.abspath(FINAL_MODEL_DIR)}")
print(f"\nSaved files:")
for file in os.listdir(FINAL_MODEL_DIR):
    print(f"   - {file}")


# Download Model (Colab vs. Local)
if IS_COLAB:
    print("\n Creating ZIP archive...")
    
    # Import shutil here for Colab-specific zipping
    
    ZIP_NAME = f"{model_folder_name.lower()}-final.zip"
    ZIP_BASE_PATH = os.path.join("/content", f"{model_folder_name.lower()}-final")
    
    shutil.make_archive(ZIP_BASE_PATH, "zip", FINAL_MODEL_DIR) 
    
    file_size_mb = os.path.getsize(f"{ZIP_BASE_PATH}.zip") / (1024*1024)
    
    print("ZIP created successfully!")
    print(f" Location: {ZIP_BASE_PATH}.zip")
    print(f" Size: {file_size_mb:.2f} MB")
    
    # files.download(f"{ZIP_BASE_PATH}.zip") # Uncomment this line for automatic download
    print("The ZIP file is saved to /content/ and can be downloaded from the Files sidebar.")

else:
    print("\n MODEL SAVED LOCALLY!")
    print(f"Your model is ready at: {os.path.abspath(FINAL_MODEL_DIR)}")


STARTING TRAINING


c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.371100,0.402951,0.876000,0.877599
2,0.291200,0.385476,0.892000,0.892659
3,0.226900,0.385064,0.892000,0.892646


c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TRAINING COMPLETE!

Evaluating final model on test set...



c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Final Test Results:
eval_loss..................... 0.3855
eval_accuracy................. 0.8920
eval_f1_macro................. 0.8927
eval_runtime.................. 48.4463
eval_samples_per_second....... 10.3210
eval_steps_per_second......... 0.6610
epoch......................... 3.0000

Saving final trained model to: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-final

Model saved to: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-final

Saved files:
   - config.json
   - model.safetensors
   - special_tokens_map.json
   - tokenizer.json
   - tokenizer_config.json
   - training_args.bin
   - vocab.txt

 MODEL SAVED LOCALLY!
Your model is ready at: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-final
